# 🔬 Semantic Search Preview 기능 실습

Azure AI Search의 **Preview 상태 기능**을 활용하여 검색 품질을 더욱 향상시키는 방법을 학습합니다.

## ⚠️ Preview 기능 주의사항

- **프로덕션 환경 사용 비권장**: Preview 기능은 테스트/개발 환경에서만 사용
- **API 버전**: `2025-11-01-preview` 이상 필요
- **변경 가능성**: Preview 기능은 GA 이전에 변경될 수 있음
- **SLA 미보장**: Service Level Agreement 적용 안 됨

## 학습 목표
1. Query Rewriting으로 쿼리 품질 개선
2. Debug 모드로 검색 내부 동작 확인
3. Preview 기능의 실무 활용 가능성 평가

---
## 1️⃣ 환경 설정

In [ ]:
import os
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.models import (
    QueryType, 
    QueryCaptionType,
    VectorizedQuery
)
from openai import AzureOpenAI

# 환경 변수 로드
load_dotenv(override=True)

# Azure AI Search 설정
SEARCH_ENDPOINT = os.getenv("SEARCH_ENDPOINT")
SEARCH_ADMIN_KEY = os.getenv("SEARCH_ADMIN_KEY")
INDEX_NAME = os.getenv("SEARCH_INDEX_NAME", "products-index")

# Azure OpenAI 설정
OPENAI_ENDPOINT = os.getenv("AZURE_OPEN_AI_ENDPOINT")
OPENAI_KEY = os.getenv("AZURE_OPEN_AI_KEY")
EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT")
API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")

credential = AzureKeyCredential(SEARCH_ADMIN_KEY)

# OpenAI Client
openai_client = AzureOpenAI(
    azure_endpoint=OPENAI_ENDPOINT,
    api_key=OPENAI_KEY,
    api_version=API_VERSION
)

# Search Client
search_client = SearchClient(
    endpoint=SEARCH_ENDPOINT,
    index_name=INDEX_NAME,
    credential=credential
)

print(f"✅ Endpoint: {SEARCH_ENDPOINT}")
print(f"✅ Index: {INDEX_NAME}")
print("⚠️ Preview API 사용: 프로덕션 환경 사용 비권장")

---
## 2️⃣ Query Rewriting (Preview)

### Query Rewriting이란?

생성형 AI 모델을 사용하여 사용자 쿼리를 **더 효과적인 대체 쿼리들로 변환**하는 기능입니다.

```
사용자 쿼리: "따뜻한 자켓"
         ↓
   Query Rewriting
         ↓
재작성된 쿼리들:
1. "보온성 좋은 겨울 자켓"
2. "따뜻하게 입을 수 있는 외투"
3. "방한 자켓"
4. "두꺼운 패딩 자켓"
5. "방풍 보온 자켓"
         ↓
    검색 실행 (원본 + 재작성된 쿼리 모두 사용)
         ↓
      더 풍부한 결과
```

### 주요 기능

- **오타/철자 수정**: "겨올 자켓" → "겨울 자켓"
- **동의어 확장**: "따뜻한" → "보온성 좋은", "방한"
- **의도 파악**: "운동할 때" → "활동적인", "신축성 있는"

### ⚠️ 제한사항

- **REST API만 지원**: Python SDK에서 직접 지원 안 함 (REST 호출 필요)
- **API 버전**: `2025-11-01-preview` 필요
- **언어 제한**: 특정 언어만 지원 (한국어: `ko-KR`)
- **최대 재작성 수**: 1~10개

### Query Rewriting 예시 (REST API)

Python SDK에서는 지원하지 않으므로, REST API를 직접 호출해야 합니다.

```python
import requests

# REST API Endpoint
url = f"{SEARCH_ENDPOINT}/indexes/{INDEX_NAME}/docs/search?api-version=2025-11-01-preview"

headers = {
    "Content-Type": "application/json",
    "api-key": SEARCH_ADMIN_KEY
}

body = {
    "search": "따뜻하고 가벼운 자켓",
    "semanticConfiguration": "products-semantic-config",
    "queryType": "semantic",
    "queryRewrites": "generative|count-5",  # 5개의 재작성 쿼리 생성
    "queryLanguage": "ko-KR",
    "debug": "queryRewrites",  # 재작성된 쿼리 확인용
    "top": 5
}

response = requests.post(url, headers=headers, json=body)
result = response.json()

# 재작성된 쿼리 확인
if "@search.debug" in result:
    rewrites = result["@search.debug"]["queryRewrites"]["text"]["rewrites"]
    print("재작성된 쿼리들:")
    for idx, rewrite in enumerate(rewrites, 1):
        print(f"{idx}. {rewrite}")
```

In [ ]:
import requests
import json

def search_with_query_rewriting(query, count=5, top=5):
    """
    Query Rewriting을 사용한 검색 (REST API)
    
    ⚠️ Preview 기능: 프로덕션 사용 비권장
    """
    
    url = f"{SEARCH_ENDPOINT}/indexes/{INDEX_NAME}/docs/search?api-version=2025-11-01-preview"
    
    headers = {
        "Content-Type": "application/json",
        "api-key": SEARCH_ADMIN_KEY
    }
    
    body = {
        "search": query,
        "semanticConfiguration": "products-semantic-config",
        "queryType": "semantic",
        "queryRewrites": f"generative|count-{count}",
        "queryLanguage": "ko-KR",
        "debug": "queryRewrites",  # 테스트용
        "top": top
    }
    
    print(f"\n{'='*80}")
    print(f" Query Rewriting 검색: '{query}'")
    print(f"{'='*80}")
    
    try:
        response = requests.post(url, headers=headers, json=body)
        response.raise_for_status()
        result = response.json()
        
        # 재작성된 쿼리 출력
        if "@search.debug" in result:
            debug_info = result["@search.debug"]
            if "queryRewrites" in debug_info:
                text_rewrites = debug_info["queryRewrites"].get("text", {})
                input_query = text_rewrites.get("inputQuery", "")
                rewrites = text_rewrites.get("rewrites", [])
                
                print(f"\n📝 원본 쿼리: {input_query}")
                print(f"\n🔄 재작성된 쿼리들 ({len(rewrites)}개):")
                print("-" * 80)
                for idx, rewrite in enumerate(rewrites, 1):
                    print(f"{idx}. {rewrite}")
        
        # 검색 결과 출력
        print(f"\n📊 검색 결과 (상위 {top}개):")
        print("-" * 80)
        
        values = result.get("value", [])
        for idx, doc in enumerate(values, 1):
            print(f"{idx}. {doc.get('title', '')[:50]}...")
            print(f"   브랜드: {doc.get('brand', '')} | 카테고리: {doc.get('category', '')}")
            print(f"   Reranker 점수: {doc.get('@search.rerankerScore', 0):.4f}")
            
            # Caption
            captions = doc.get('@search.captions', [])
            if captions and len(captions) > 0:
                caption_text = captions[0].get('text', '')
                if caption_text:
                    print(f"   💬 {caption_text[:100]}...")
            print()
        
        return result
        
    except requests.exceptions.HTTPError as e:
        print(f"\n❌ HTTP Error: {e}")
        print(f"Response: {e.response.text}")
        return None
    except Exception as e:
        print(f"\n❌ Error: {e}")
        return None

print("✅ Query Rewriting 함수 정의 완료")
print("⚠️ 이 기능은 REST API를 직접 호출합니다 (Python SDK 미지원)")

### 실습 1: 오타/철자 수정

In [ ]:
# 시나리오 1: 오타가 있는 쿼리
result = search_with_query_rewriting("따뜻하고 가버운 겨율 자켓", count=5, top=5)

print("\n💡 분석:")
print("   - 원본: '가버운' (오타), '겨율' (오타)")
print("   - Query Rewriting이 올바른 철자로 수정")
print("   → 오타가 있어도 정확한 검색 결과 제공")

### 실습 2: 동의어 확장

In [ ]:
# 시나리오 2: 동의어 확장이 필요한 쿼리
result = search_with_query_rewriting("운동할 때 입기 좋은 옷", count=5, top=5)

print("\n💡 분석:")
print("   - 원본: '운동할 때', '입기 좋은'")
print("   - 재작성: '활동적인', '신축성', '스포츠웨어' 등 동의어 확장")
print("   → 더 다양한 관련 상품 검색 가능")

### 실습 3: 복합 의도 파악

In [ ]:
# 시나리오 3: 복합 의도 쿼리
result = search_with_query_rewriting("아이 생일 선물용 유아용품", count=5, top=5)

print("\n💡 분석:")
print("   - 원본: '생일 선물용'이라는 의도")
print("   - 재작성: '선물하기 좋은', '기념일용', '특별한 날' 등으로 확장")
print("   → 선물 목적에 맞는 상품 더 잘 찾음")

### 실습 4: 품질 중심 검색

In [ ]:
# 시나리오 4: 품질/만족도 중심 쿼리
result = search_with_query_rewriting("품질 좋고 평점 높은 제품", count=5, top=5)

print("\n💡 분석:")
print("   - 원본: '품질 좋고', '평점 높은'")
print("   - 재작성: '고품질', '만족도 높은', '인기 있는' 등으로 확장")
print("   → 품질 평가가 좋은 상품 우선 노출")

---
## 3️⃣ Debug 모드 (Preview)

### Debug 모드란?

검색 엔진의 **내부 동작을 확인**할 수 있는 기능입니다.

### Debug 옵션

| Debug 모드 | 확인 내용 |
|-----------|----------|
| `"semantic"` | Semantic Ranking 내부 동작 |
| `"queryRewrites"` | 재작성된 쿼리들 |
| 모두 | 전체 디버그 정보 |

### ⚠️ 주의사항

- **프로덕션 사용 금지**: 성능 저하
- **테스트 전용**: 개발/디버깅 목적만 사용
- **응답 크기 증가**: 디버그 정보로 인해 응답 크기 증가

In [ ]:
def search_with_debug(query, debug_mode="semantic", top=3):
    """
    Debug 모드로 검색 (REST API)
    
    ⚠️ Preview 기능: 프로덕션 사용 금지
    """
    
    url = f"{SEARCH_ENDPOINT}/indexes/{INDEX_NAME}/docs/search?api-version=2025-11-01-preview"
    
    headers = {
        "Content-Type": "application/json",
        "api-key": SEARCH_ADMIN_KEY
    }
    
    body = {
        "search": query,
        "semanticConfiguration": "products-semantic-config",
        "queryType": "semantic",
        "debug": debug_mode,  # Debug 모드 활성화
        "top": top
    }
    
    print(f"\n{'='*80}")
    print(f" Debug 모드 검색: '{query}'")
    print(f" Debug 옵션: {debug_mode}")
    print(f"{'='*80}")
    
    try:
        response = requests.post(url, headers=headers, json=body)
        response.raise_for_status()
        result = response.json()
        
        # Debug 정보 출력
        if "@search.debug" in result:
            print("\n🔍 Debug 정보:")
            print("-" * 80)
            debug_info = result["@search.debug"]
            print(json.dumps(debug_info, indent=2, ensure_ascii=False))
        
        # 검색 결과 요약
        print(f"\n📊 검색 결과 요약:")
        print("-" * 80)
        values = result.get("value", [])
        print(f"총 {len(values)}개 결과")
        
        for idx, doc in enumerate(values, 1):
            print(f"{idx}. {doc.get('title', '')[:40]}... (Reranker: {doc.get('@search.rerankerScore', 0):.4f})")
        
        return result
        
    except Exception as e:
        print(f"\n❌ Error: {e}")
        return None

print("✅ Debug 함수 정의 완료")
print("⚠️ Debug 모드는 테스트 목적으로만 사용하세요")

### Debug 실습: Semantic Ranking 내부 확인

In [ ]:
# Semantic Ranking의 내부 동작 확인
result = search_with_debug("따뜻하고 가벼운 자켓", debug_mode="semantic", top=3)

print("\n💡 분석:")
print("   - Debug 정보로 Semantic Ranking의 내부 동작 확인 가능")
print("   - 프로덕션에서는 사용하지 말 것 (성능 저하)")

---
## 4️⃣ Query Rewriting + Debug 결합

두 Preview 기능을 함께 사용하여 **재작성된 쿼리와 검색 내부 동작을 동시에 확인**할 수 있습니다.

In [ ]:
def search_with_full_debug(query, rewrite_count=3, top=3):
    """
    Query Rewriting + Debug 모드 결합
    
    ⚠️ Preview 기능 조합: 테스트 전용
    """
    
    url = f"{SEARCH_ENDPOINT}/indexes/{INDEX_NAME}/docs/search?api-version=2025-11-01-preview"
    
    headers = {
        "Content-Type": "application/json",
        "api-key": SEARCH_ADMIN_KEY
    }
    
    body = {
        "search": query,
        "semanticConfiguration": "products-semantic-config",
        "queryType": "semantic",
        "queryRewrites": f"generative|count-{rewrite_count}",
        "queryLanguage": "ko-KR",
        "debug": "queryRewrites",  # 재작성 쿼리 확인
        "top": top
    }
    
    print(f"\n{'='*80}")
    print(f" 🔬 Full Debug: Query Rewriting + Semantic Ranking")
    print(f" 검색어: '{query}'")
    print(f"{'='*80}")
    
    try:
        response = requests.post(url, headers=headers, json=body)
        response.raise_for_status()
        result = response.json()
        
        # 전체 Debug 정보 출력
        if "@search.debug" in result:
            print("\n📋 전체 Debug 정보:")
            print("-" * 80)
            print(json.dumps(result["@search.debug"], indent=2, ensure_ascii=False))
        
        # 검색 결과
        print(f"\n📊 검색 결과:")
        print("-" * 80)
        values = result.get("value", [])
        for idx, doc in enumerate(values, 1):
            print(f"{idx}. {doc.get('title', '')[:50]}...")
            print(f"   Reranker: {doc.get('@search.rerankerScore', 0):.4f}")
        
        return result
        
    except Exception as e:
        print(f"\n❌ Error: {e}")
        return None

print("✅ Full Debug 함수 정의 완료")

In [ ]:
# Full Debug 실행
result = search_with_full_debug("데일리로 입기 좋은 편한 옷", rewrite_count=3, top=3)

print("\n💡 분석:")
print("   - Query Rewriting: 쿼리가 어떻게 재작성되는지 확인")
print("   - Semantic Ranking: 어떤 문서가 높은 점수를 받는지 확인")
print("   - 두 기능의 시너지로 검색 품질 향상 과정 이해 가능")

---
## 5️⃣ Preview 기능 vs GA 기능 비교

### 기능별 상태 비교

| 기능 | 상태 | 프로덕션 사용 | SDK 지원 | API 버전 |
|------|------|--------------|----------|----------|
| **Semantic Re-Ranking** | ✅ GA | ✅ 권장 | ✅ 지원 | 2023-11-01 |
| **Captions** | ✅ GA | ✅ 권장 | ✅ 지원 | 2023-11-01 |
| **Answers** | ✅ GA | ✅ 권장 | ✅ 지원 | 2023-11-01 |
| **Query Rewriting** | 🔬 Preview | ⚠️ 비권장 | ❌ 미지원 | 2025-11-01-preview |
| **Debug 모드** | 🔬 Preview | ⚠️ 비권장 | ❌ 미지원 | 2025-11-01-preview |

### Preview 기능 사용 시 고려사항

#### ✅ 사용 적합한 경우
- **개발/테스트 환경**
- **POC (Proof of Concept)**
- **새 기능 평가 및 테스트**

#### ❌ 사용 부적합한 경우
- **프로덕션 환경**
- **SLA가 필요한 서비스**
- **안정성이 중요한 서비스**

### 비용 비교

| 기능 | 비용 | 추가 비용 |
|------|------|----------|
| Hybrid Search (RRF) | 기본 | - |
| + Semantic Re-Ranking | +$0.50/1000 쿼리 | 추가 |
| + Query Rewriting | +α (Preview 요금 미정) | 추가 |
| + Debug 모드 | 무료 (성능 저하) | - |

---
## 6️⃣ 정리 및 권장사항

### ✅ 학습한 내용

1. **Query Rewriting (Preview)**
   - 생성형 AI로 쿼리를 재작성하여 검색 품질 개선
   - 오타 수정, 동의어 확장, 의도 파악
   - REST API 직접 호출 필요 (Python SDK 미지원)

2. **Debug 모드 (Preview)**
   - 검색 엔진 내부 동작 확인
   - Semantic Ranking, Query Rewriting 등의 내부 로직 확인
   - 테스트/디버깅 목적으로만 사용

3. **두 기능의 조합**
   - Query Rewriting으로 쿼리 개선
   - Debug 모드로 내부 동작 확인
   - 검색 품질 향상 과정 이해

### 📋 실무 권장사항

#### 개발 단계
```
1. GA 기능으로 기본 구현
   → Hybrid Search + Semantic Re-Ranking + Captions/Answers
   
2. Preview 기능으로 테스트
   → Query Rewriting, Debug 모드
   
3. 성능/품질 평가
   → Preview 기능의 실제 효과 측정
   
4. GA 전환 대기
   → Preview 기능이 GA로 전환될 때까지 대기
```

#### 프로덕션 배포
```
✅ 사용: GA 기능만 사용
   - Semantic Re-Ranking
   - Captions
   - Answers
   
❌ 제외: Preview 기능 제외
   - Query Rewriting → GA 전환 후 고려
   - Debug 모드 → 완전히 제거
```

### ⚠️ 최종 주의사항

- **Preview 기능은 변경될 수 있음**: API 스펙, 동작 방식 변경 가능
- **SLA 미보장**: 장애 발생 시 보상 없음
- **성능 저하 가능**: Debug 모드는 특히 성능에 영향
- **비용 불확실**: Preview 요금은 GA 전환 시 변경될 수 있음

### 💡 핵심 요약

```
Preview 기능 = 미래 기능 체험
             ↓
        테스트 환경에서 평가
             ↓
        GA 전환 시 프로덕션 도입 검토
```

In [ ]:
print("✅ Preview 기능 실습 완료!")
print("\n📋 다음 단계:")
print("1. GA 기능 (01-semantic_reranking.ipynb) 완전 숙지")
print("2. Preview 기능은 테스트 환경에서만 활용")
print("3. GA 전환 소식 모니터링 (Microsoft Learn, Azure Updates)")
print("\n⚠️ 프로덕션 환경에서는 GA 기능만 사용하세요!")